# Notebook 08 -- Quantum Phase Estimation and Textbook Algorithms

The earliest quantum algorithms were designed not to solve practical problems,
but to prove that quantum computers can outperform classical ones in
well-defined settings. These **textbook algorithms** operate in the **oracle
model**: they treat a function $f$ as a black box and ask how many queries
are needed to learn some property of $f$.

In this notebook we will implement four foundational algorithms:

1. **Deutsch-Jozsa** -- distinguishes constant from balanced functions in 1 query.
2. **Bernstein-Vazirani** -- recovers a hidden bitstring in 1 query.
3. **Simon's algorithm** -- finds a hidden period with exponentially fewer queries.
4. **Quantum Phase Estimation (QPE)** -- extracts the eigenvalue phase of a unitary.

Each algorithm demonstrates a provable quantum speedup over the best
classical approach, establishing the theoretical foundation for quantum
computing's power.

In [ ]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/qpe"
	"github.com/splch/goqu/algorithm/textbook"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/gate"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## The Deutsch-Jozsa Algorithm

**Problem:** Given a black-box function $f: \{0,1\}^n \to \{0,1\}$ that is
promised to be either **constant** (same output for all inputs) or **balanced**
(outputs 0 for exactly half the inputs and 1 for the other half), determine
which type it is.

**Classical lower bound:** Any deterministic classical algorithm needs
$2^{n-1} + 1$ queries in the worst case -- it must check just over half the
inputs before it can be certain.

**Quantum solution:** The Deutsch-Jozsa algorithm answers with **a single
query** by exploiting interference:

1. Prepare all input qubits in superposition with H gates.
2. Apply the oracle once.
3. Apply H gates again and measure.

If $f$ is constant, all input qubits measure $|0\rangle$ (constructive
interference on the all-zeros outcome). If $f$ is balanced, at least one
qubit measures $|1\rangle$ (destructive interference eliminates the all-zeros
outcome).

In [ ]:
%%
ctx := context.Background()

// Run Deutsch-Jozsa with a constant oracle (f(x) = 0 for all x)
result, err := textbook.DeutschJozsa(ctx, textbook.DJConfig{
	NumQubits: 3,
	Oracle:    textbook.ConstantOracle(0),
	Shots:     1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Constant Oracle (f(x) = 0) ===")
fmt.Println("Circuit:")
fmt.Println(draw.String(result.Circuit))
fmt.Println("Measurement counts:", result.Counts)
fmt.Printf("Is constant: %v\n", result.IsConstant)
fmt.Println("\nAll qubits measure 0 -- constructive interference on |000>.")

In [ ]:
%%
ctx := context.Background()

// Run Deutsch-Jozsa with a balanced oracle (mask = 0b101)
// This oracle computes f(x) = popcount(x AND 101) mod 2
result, err := textbook.DeutschJozsa(ctx, textbook.DJConfig{
	NumQubits: 3,
	Oracle:    textbook.BalancedOracle(0b101),
	Shots:     1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Balanced Oracle (mask = 101) ===")
fmt.Println("Measurement counts:", result.Counts)
fmt.Printf("Is constant: %v\n", result.IsConstant)
fmt.Println("\nThe all-zeros outcome is absent -- the algorithm detects 'balanced'.")

In [ ]:
%%
ctx := context.Background()

// Histogram comparing constant vs balanced results
constResult, _ := textbook.DeutschJozsa(ctx, textbook.DJConfig{
	NumQubits: 3,
	Oracle:    textbook.ConstantOracle(0),
	Shots:     1000,
})
balResult, _ := textbook.DeutschJozsa(ctx, textbook.DJConfig{
	NumQubits: 3,
	Oracle:    textbook.BalancedOracle(0b101),
	Shots:     1000,
})

svgConst := viz.Histogram(constResult.Counts, viz.WithTitle("Deutsch-Jozsa: Constant Oracle"))
svgBal := viz.Histogram(balResult.Counts, viz.WithTitle("Deutsch-Jozsa: Balanced Oracle"))
gonbui.DisplayHTML(svgConst)
gonbui.DisplayHTML(svgBal)

## The Bernstein-Vazirani Algorithm

**Problem:** Given an oracle for $f(x) = s \cdot x \mod 2$ (the dot product
of a hidden string $s$ with the input $x$, modulo 2), find $s$.

**Classical lower bound:** A classical algorithm needs $n$ queries -- one for
each bit of $s$ (query with $x = e_i$ to learn $s_i$).

**Quantum solution:** Bernstein-Vazirani recovers the entire $n$-bit secret
in **a single query**. The circuit is identical to Deutsch-Jozsa (Hadamard,
oracle, Hadamard, measure), but the measurement outcome directly encodes
the secret string $s$.

The key insight: the Hadamard transform converts the phase kickback from
the oracle into a bitstring in the computational basis.

In [ ]:
%%
ctx := context.Background()

// Run Bernstein-Vazirani to find the secret string s = 1011 (decimal 11)
result, err := textbook.BernsteinVazirani(ctx, textbook.BVConfig{
	NumQubits: 4,
	Secret:    0b1011, // secret = "1011"
	Shots:     1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Bernstein-Vazirani ===")
fmt.Println("Circuit:")
fmt.Println(draw.String(result.Circuit))
fmt.Printf("Secret found: %04b (decimal %d)\n", result.Secret, result.Secret)
fmt.Println("Measurement counts:", result.Counts)
fmt.Println("\nThe secret string 1011 is recovered in a single query!")

In [ ]:
%%
ctx := context.Background()

result, _ := textbook.BernsteinVazirani(ctx, textbook.BVConfig{
	NumQubits: 4,
	Secret:    0b1011,
	Shots:     1000,
})

svg := viz.Histogram(result.Counts, viz.WithTitle("Bernstein-Vazirani: Secret = 1011"))
gonbui.DisplayHTML(svg)

## Simon's Algorithm

**Problem:** Given an oracle for $f: \{0,1\}^n \to \{0,1\}^n$ with the
promise that there exists a secret period $s$ such that $f(x) = f(y)$ if and
only if $x = y$ or $x = y \oplus s$, find $s$.

**Classical lower bound:** Any classical algorithm requires $\Omega(2^{n/2})$
queries (birthday paradox bound).

**Quantum solution:** Simon's algorithm needs only $O(n)$ quantum queries.
Each query produces a random value $y$ satisfying $y \cdot s = 0 \mod 2$.
After collecting $n-1$ linearly independent equations over GF(2), Gaussian
elimination recovers $s$.

Simon's algorithm is historically significant as the inspiration for Shor's
algorithm -- both use the same pattern of applying a function in superposition
and then performing a Fourier-like transform to extract hidden structure.

In [ ]:
%%
ctx := context.Background()

// Run Simon's algorithm with secret period s = 110 (decimal 6)
result, err := textbook.Simon(ctx, textbook.SimonConfig{
	NumQubits: 3,
	Oracle:    textbook.TwoToOneOracle(0b110, 3),
	Shots:     1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Simon's Algorithm ===")
fmt.Println("Circuit (last round):")
fmt.Println(draw.String(result.Circuit))
fmt.Printf("Recovered period: %03b (decimal %d)\n", result.Period, result.Period)
fmt.Printf("Equations collected: %d\n", len(result.Equations))
for i, eq := range result.Equations {
	fmt.Printf("  y_%d = %03b  (y . s = 0 mod 2)\n", i, eq)
}

## Quantum Phase Estimation (QPE)

**Problem:** Given a unitary operator $U$ and one of its eigenstates
$|\psi\rangle$ satisfying $U|\psi\rangle = e^{2\pi i \varphi}|\psi\rangle$,
estimate the phase $\varphi$.

QPE is the most important subroutine in quantum computing. It underpins:
- **Shor's algorithm** (factoring via order-finding)
- **Quantum chemistry** (energy estimation of molecular Hamiltonians)
- **HHL algorithm** (linear systems solver)

**How it works:**

1. Prepare a **phase register** of $t$ qubits in superposition.
2. Apply controlled-$U^{2^k}$ operations to transfer phase information.
3. Apply the **inverse Quantum Fourier Transform** (QFT$^\dagger$) to convert
   phase into a binary fraction readable by measurement.

With $t$ phase bits, QPE estimates $\varphi$ to $t$ bits of precision:
$\hat{\varphi} = m / 2^t$ where $m$ is the measured integer.

For the **T gate** ($\pi/8$ gate), the eigenvalue on $|1\rangle$ is
$e^{i\pi/4} = e^{2\pi i \cdot 1/8}$, so the phase is $\varphi = 1/8 = 0.125$.

In [ ]:
%%
ctx := context.Background()

// QPE on the T gate: T|1> = exp(i*pi/4)|1> = exp(2*pi*i * 1/8)|1>
// So the phase is 1/8 = 0.125
eigenCircuit, err := builder.New("eigen", 1).X(0).Build()
if err != nil {
	panic(err)
}

result, err := qpe.Run(ctx, qpe.Config{
	Unitary:      gate.T,
	NumPhaseBits: 3,
	EigenState:   eigenCircuit,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

fmt.Println("=== Quantum Phase Estimation ===")
fmt.Printf("Unitary: T gate (phase = pi/4)\n")
fmt.Printf("Phase bits: %d\n", result.PhaseRegBits)
fmt.Printf("Estimated phase: %.4f (expected: 0.125)\n", result.Phase)
fmt.Printf("Phase as fraction: 1/8 = %.4f\n", 1.0/8.0)
fmt.Println("Measurement counts:", result.Counts)

In [ ]:
%%
ctx := context.Background()

eigenCircuit, _ := builder.New("eigen", 1).X(0).Build()

result, _ := qpe.Run(ctx, qpe.Config{
	Unitary:      gate.T,
	NumPhaseBits: 3,
	EigenState:   eigenCircuit,
	Shots:        1000,
})

fmt.Println("QPE Circuit:")
fmt.Println(draw.String(result.Circuit))

svg := viz.Histogram(result.Counts, viz.WithTitle("QPE: T Gate (expected phase = 0.125)"))
gonbui.DisplayHTML(svg)

## State Preparation

Many quantum algorithms require initializing qubits in a specific state
beyond the standard $|0\rangle$. The `gate.MustStatePrep` function creates
a gate that prepares any normalized state from a list of amplitudes.

Internally, it uses a Mottonen decomposition -- a sequence of uniformly
controlled RY and RZ rotations that systematically constructs the target
state from $|0\rangle$.

Let's prepare the state $\frac{1}{2}|0\rangle + \frac{\sqrt{3}}{2}|1\rangle$,
which has $P(0) = 1/4$ and $P(1) = 3/4$.

In [ ]:
%%
// Prepare a known state using gate.MustStatePrep
// Target: (1/2)|0> + (sqrt(3)/2)|1>  =>  P(0)=0.25, P(1)=0.75
amplitudes := []complex128{
	complex(0.5, 0),             // amplitude for |0>
	complex(math.Sqrt(3)/2, 0),  // amplitude for |1>
}

spGate := gate.MustStatePrep(amplitudes)

c, err := builder.New("state-prep", 1).
	Apply(spGate, 0).
	MeasureAll().
	Build()
if err != nil {
	panic(err)
}

sim := statevector.New(1)
counts, err := sim.Run(c, 2000)
if err != nil {
	panic(err)
}

fmt.Println("State preparation: (1/2)|0> + (sqrt(3)/2)|1>")
fmt.Println("Expected: P(0) = 0.25, P(1) = 0.75")
fmt.Println("Measured counts (2000 shots):", counts)
for outcome, count := range counts {
	fmt.Printf("  |%s> : %d (%.1f%%)\n", outcome, count, float64(count)/20.0)
}

svg := viz.Histogram(counts, viz.WithTitle("State Preparation: P(0)=0.25, P(1)=0.75"))
gonbui.DisplayHTML(svg)

## Predict, Then Verify

**Question:** The T gate applies a phase of $e^{i\pi/4}$ to $|1\rangle$,
which corresponds to $e^{2\pi i \cdot 1/8}$, giving phase $\varphi = 1/8$.

What phase will QPE extract from the **S gate**?

**Prediction:** The S gate applies $e^{i\pi/2}$ to $|1\rangle$:

$$S|1\rangle = e^{i\pi/2}|1\rangle = e^{2\pi i \cdot 1/4}|1\rangle$$

So the phase should be $\varphi = 1/4 = 0.25$. With 3 phase bits, this is
exactly representable as $2/8 = 0.010$ in binary, so QPE should recover it
perfectly.

In [ ]:
%%
ctx := context.Background()

// Verify: QPE on the S gate should give phase = 0.25
eigenCircuit, _ := builder.New("eigen", 1).X(0).Build()

result, err := qpe.Run(ctx, qpe.Config{
	Unitary:      gate.S,
	NumPhaseBits: 3,
	EigenState:   eigenCircuit,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("S gate -- Estimated phase: %.4f (expected: 0.2500)\n", result.Phase)
fmt.Printf("Match: %v\n", math.Abs(result.Phase-0.25) < 0.01)
fmt.Println("Counts:", result.Counts)
fmt.Println("\nPrediction confirmed: S gate phase = 1/4 = 0.25.")

---

## Exercises

### Exercise 1 -- Bernstein-Vazirani with a Different Secret

Run the Bernstein-Vazirani algorithm with 5 qubits and secret $s = 10110$
(decimal 22). Verify that the algorithm recovers the secret in a single query.

In [ ]:
%%
// Exercise 1: BV with 5 qubits, secret = 10110 (22)
ctx := context.Background()

result, err := textbook.BernsteinVazirani(ctx, textbook.BVConfig{
	NumQubits: 5,
	Secret:    0b10110, // 22
	Shots:     1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("Secret found: %05b (decimal %d)\n", result.Secret, result.Secret)
fmt.Printf("Correct: %v\n", result.Secret == 22)
fmt.Println("Counts:", result.Counts)

svg := viz.Histogram(result.Counts, viz.WithTitle("Exercise 1: BV Secret = 10110"))
gonbui.DisplayHTML(svg)

### Exercise 2 -- QPE with Higher Precision

Run QPE on the T gate with **5 phase bits** instead of 3. Because the true
phase $1/8 = 0.00100$ in binary is exactly representable with 3 bits,
increasing to 5 bits should still produce a sharp peak at the same value.
Compare the measurement distributions.

In [ ]:
%%
// Exercise 2: QPE on T gate with 5 phase bits for higher precision
ctx := context.Background()

eigenCircuit, _ := builder.New("eigen", 1).X(0).Build()

result3, _ := qpe.Run(ctx, qpe.Config{
	Unitary:      gate.T,
	NumPhaseBits: 3,
	EigenState:   eigenCircuit,
	Shots:        1000,
})

result5, _ := qpe.Run(ctx, qpe.Config{
	Unitary:      gate.T,
	NumPhaseBits: 5,
	EigenState:   eigenCircuit,
	Shots:        1000,
})

fmt.Printf("3 phase bits -- phase: %.6f\n", result3.Phase)
fmt.Printf("5 phase bits -- phase: %.6f\n", result5.Phase)
fmt.Printf("Expected:               0.125000\n")
fmt.Println("\n3-bit counts:", result3.Counts)
fmt.Println("5-bit counts:", result5.Counts)

svg3 := viz.Histogram(result3.Counts, viz.WithTitle("QPE T Gate: 3 Phase Bits"))
svg5 := viz.Histogram(result5.Counts, viz.WithTitle("QPE T Gate: 5 Phase Bits"))
gonbui.DisplayHTML(svg3)
gonbui.DisplayHTML(svg5)

## Key Takeaways

1. **Oracle model algorithms** demonstrate quantum speedups by treating
   functions as black boxes. The speedup is provable because we can count
   queries exactly.

2. **Deutsch-Jozsa** achieves an exponential query speedup: 1 quantum query
   vs. $2^{n-1}+1$ classical queries to distinguish constant from balanced.

3. **Bernstein-Vazirani** achieves a linear query speedup: 1 quantum query
   vs. $n$ classical queries to recover a hidden bitstring.

4. **Simon's algorithm** achieves an exponential speedup: $O(n)$ quantum
   queries vs. $\Omega(2^{n/2})$ classical queries to find a hidden period.
   It directly inspired Shor's factoring algorithm.

5. **Quantum Phase Estimation** extracts the eigenvalue phase of a unitary
   operator using the inverse QFT. With $t$ ancilla qubits, it achieves
   $t$ bits of precision. QPE is the key subroutine in Shor's algorithm,
   quantum chemistry, and the HHL algorithm.

6. **State preparation** via `gate.MustStatePrep` uses Mottonen decomposition
   to initialize arbitrary quantum states from amplitude vectors.

---

**Next up:** Notebook 09 — Grover's Search Algorithm, where we'll harness amplitude amplification for unstructured search.